In [ ]:
!wget -O Task09_Spleen.tar "https://zenodo.org/records/15924508/files/Task09_Spleen.tar?download=1"
!tar -xf Task09_Spleen.tar
!ls Task09_Spleen

--2026-02-27 23:10:33--  https://zenodo.org/records/15924508/files/Task09_Spleen.tar?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.98.114, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1610352640 (1.5G) [application/octet-stream]
Saving to: ‘Task09_Spleen.tar’

Task09_Spleen.tar   100%[===================>]   1.50G  18.3MB/s    in 87s     

2026-02-27 23:12:01 (17.6 MB/s) - ‘Task09_Spleen.tar’ saved [1610352640/1610352640]

dataset.json  imagesTr	imagesTs  labelsTr


In [ ]:
!ls Task09_Spleen

dataset.json  imagesTr	imagesTs  labelsTr


In [ ]:
!pip install nibabel tqdm opencv-python

In [ ]:
!ls Task09_Spleen
!ls Task09_Spleen/imagesTr | head
!ls Task09_Spleen/labelsTr | head

dataset.json  imagesTr	imagesTs  labelsTr
spleen_10.nii.gz
spleen_12.nii.gz
spleen_13.nii.gz
spleen_14.nii.gz
spleen_16.nii.gz
spleen_17.nii.gz
spleen_18.nii.gz
spleen_19.nii.gz
spleen_20.nii.gz
spleen_21.nii.gz
spleen_10.nii.gz
spleen_12.nii.gz
spleen_13.nii.gz
spleen_14.nii.gz
spleen_16.nii.gz
spleen_17.nii.gz
spleen_18.nii.gz
spleen_19.nii.gz
spleen_20.nii.gz
spleen_21.nii.gz


In [ ]:
import os
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm

image_dir = "/content/Task09_Spleen/imagesTr"
label_dir = "/content/Task09_Spleen/labelsTr"

assert os.path.isdir(image_dir), f"Missing image_dir: {image_dir}"
assert os.path.isdir(label_dir), f"Missing label_dir: {label_dir}"

print("Example images:", sorted(os.listdir(image_dir))[:3])
print("Example labels:", sorted(os.listdir(label_dir))[:3])

out_img = "/content/data/images"
out_msk = "/content/data/masks"
os.makedirs(out_img, exist_ok=True)
os.makedirs(out_msk, exist_ok=True)

def window_ct(x, level=50, width=400):
    lo = level - width/2
    hi = level + width/2
    x = np.clip(x, lo, hi)
    x = (x - lo) / (hi - lo + 1e-8)
    return (x * 255).astype(np.uint8)

def image_to_label_name(img_name: str) -> str:
    # Common MSD convention: case_XXXX_0000.nii.gz -> case_XXXX.nii.gz
    if img_name.endswith("_0000.nii.gz"):
        return img_name.replace("_0000.nii.gz", ".nii.gz")
    return img_name  # already matching

saved = 0
skipped_missing_label = 0

for img_fname in tqdm(sorted(os.listdir(image_dir))):
    if not img_fname.endswith(".nii.gz"):
        continue

    lbl_fname = image_to_label_name(img_fname)
    img_path = os.path.join(image_dir, img_fname)
    lbl_path = os.path.join(label_dir, lbl_fname)

    if not os.path.exists(lbl_path):
        skipped_missing_label += 1
        continue

    img = nib.load(img_path).get_fdata()
    msk = nib.load(lbl_path).get_fdata()

    # axial slices
    for z in range(img.shape[2]):
        m = msk[:, :, z]
        if m.sum() == 0:
            continue

        x = window_ct(img[:, :, z])
        x_rgb = np.stack([x, x, x], axis=-1)

        stem = img_fname.replace(".nii.gz", "").replace("_0000", "")
        out_name = f"{stem}_z{z:03d}.png"

        cv2.imwrite(os.path.join(out_img, out_name), x_rgb)
        cv2.imwrite(os.path.join(out_msk, out_name), (m > 0).astype(np.uint8) * 255)
        saved += 1

print("✅ Saved non-empty slices:", saved)
print("⚠️ Images skipped (missing label match):", skipped_missing_label)
print("Output image folder:", out_img, "count:", len(os.listdir(out_img)))
print("Output mask  folder:", out_msk, "count:", len(os.listdir(out_msk)))

Example images: ['._spleen_10.nii.gz', '._spleen_12.nii.gz', '._spleen_13.nii.gz']
Example labels: ['._spleen_10.nii.gz', '._spleen_12.nii.gz', '._spleen_13.nii.gz']


  0%|          | 0/82 [00:00<?, ?it/s]


ImageFileError: File /content/Task09_Spleen/imagesTr/._spleen_10.nii.gz is not a gzip file

In [ ]:
files = [f for f in sorted(os.listdir(image_dir))
         if f.endswith(".nii.gz") and not f.startswith("._")]

saved = 0

for fname in tqdm(files):
    img = nib.load(os.path.join(image_dir, fname)).get_fdata()
    msk = nib.load(os.path.join(label_dir, fname)).get_fdata()

    for z in range(img.shape[2]):
        m = msk[:, :, z]
        if m.sum() == 0:
            continue

        x = window_ct(img[:, :, z])
        x_rgb = np.stack([x, x, x], axis=-1)

        out_name = f"{fname.replace('.nii.gz','')}_z{z:03d}.png"
        cv2.imwrite(os.path.join(out_img, out_name), x_rgb)
        cv2.imwrite(os.path.join(out_msk, out_name), (m > 0).astype(np.uint8) * 255)

        saved += 1

print("✅ Saved slices:", saved)

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install matplotlib

# Download SAM ViT-B checkpoint (fast, Colab-friendly)
!wget -q -O sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import os, random
import cv2
import matplotlib.pyplot as plt
import numpy as np

img_dir = "data/images"
msk_dir = "data/masks"

f = random.choice(os.listdir(img_dir))
img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f)), cv2.COLOR_BGR2RGB)
msk = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.title("CT slice"); plt.imshow(img); plt.axis("off")
plt.subplot(1,2,2); plt.title("GT mask"); plt.imshow(msk, cmap="gray"); plt.axis("off")
plt.show()

print("File:", f, " Mask nonzero:", int((msk>0).sum()))

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to(device=device)
predictor = SamPredictor(sam)

def mask_to_box(mask):
    ys, xs = np.where(mask > 0)
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return np.array([x0, y0, x1, y1])

def dice_iou(pred, gt):
    pred = (pred > 0).astype(np.uint8)
    gt = (gt > 0).astype(np.uint8)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    dice = (2*inter) / (pred.sum() + gt.sum() + 1e-8)
    iou = inter / (union + 1e-8)
    return float(dice), float(iou)

# Evaluate a small batch first (fast test)
import pandas as pd
from tqdm import tqdm

img_files = sorted(os.listdir(img_dir))[:50]  # change to more later
rows = []

for f in tqdm(img_files):
    img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f)), cv2.COLOR_BGR2RGB)
    gt  = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)

    box = mask_to_box(gt)

    predictor.set_image(img)
    masks, scores, _ = predictor.predict(
        box=box,
        multimask_output=True
    )
    # Choose best scoring mask
    best = masks[np.argmax(scores)]

    d, j = dice_iou(best, gt)
    rows.append({"file": f, "dice": d, "iou": j})

df = pd.DataFrame(rows)
print(df[["dice","iou"]].describe())

In [ ]:
img_files = sorted(os.listdir(img_dir))
rows = []

for f in tqdm(img_files):
    img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f)), cv2.COLOR_BGR2RGB)
    gt  = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)

    box = mask_to_box(gt)
    predictor.set_image(img)

    masks, scores, _ = predictor.predict(box=box, multimask_output=True)
    best = masks[np.argmax(scores)]

    d, j = dice_iou(best, gt)
    rows.append({"file": f, "dice": d, "iou": j})

df = pd.DataFrame(rows)
df.to_csv("spleen_sam_boxprompt_metrics.csv", index=False)

print("✅ Saved: spleen_sam_boxprompt_metrics.csv")
print(df[["dice","iou"]].describe())

In [ ]:
import pandas as pd

df = pd.read_csv("spleen_sam_boxprompt_metrics.csv")
fail_05 = (df["dice"] < 0.5).mean()
fail_01 = (df["dice"] < 0.1).mean()

print("Failure rate (Dice < 0.5):", round(fail_05*100, 2), "%")
print("Severe failure (Dice < 0.1):", round(fail_01*100, 2), "%")
print("Worst 10 examples:")
print(df.sort_values("dice").head(10))

In [ ]:
import os, cv2, numpy as np, matplotlib.pyplot as plt
import pandas as pd

img_dir = "data/images"
msk_dir = "data/masks"
df = pd.read_csv("spleen_sam_boxprompt_metrics.csv").sort_values("dice").head(8)

def overlay(img, mask, alpha=0.4):
    out = img.copy()
    red = np.zeros_like(img); red[...,0] = 255
    m = (mask > 0)
    out[m] = (1-alpha)*out[m] + alpha*red[m]
    return out.astype(np.uint8)

plt.figure(figsize=(16,8))
for idx, row in enumerate(df.itertuples(), 1):
    f = row.file
    img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f)), cv2.COLOR_BGR2RGB)
    gt  = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)

    # load SAM prediction again using your predictor
    predictor.set_image(img)
    ys, xs = np.where(gt > 0)
    box = np.array([xs.min(), ys.min(), xs.max(), ys.max()])
    masks, scores, _ = predictor.predict(box=box, multimask_output=True)
    pred = masks[np.argmax(scores)].astype(np.uint8)*255

    ov = overlay(img, pred)
    plt.subplot(2,4,idx)
    plt.title(f"Dice={row.dice:.3f}")
    plt.imshow(ov); plt.axis("off")

plt.tight_layout()
plt.savefig("worst_cases_overlay.png", dpi=200)
print("✅ Saved worst_cases_overlay.png")

In [ ]:
import os, cv2, numpy as np, pandas as pd
from tqdm import tqdm

img_dir = "data/images"
msk_dir = "data/masks"

def mask_to_box(mask):
    ys, xs = np.where(mask > 0)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()])

def dice_iou(pred, gt):
    pred = (pred > 0).astype(np.uint8)
    gt = (gt > 0).astype(np.uint8)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    dice = (2*inter) / (pred.sum() + gt.sum() + 1e-8)
    iou = inter / (union + 1e-8)
    return float(dice), float(iou)

# --- perturbations ---
def add_gaussian_noise(img, sigma):
    x = img.astype(np.float32)
    n = np.random.normal(0, sigma, x.shape).astype(np.float32)
    y = np.clip(x + n, 0, 255)
    return y.astype(np.uint8)

def gaussian_blur(img, k):
    return cv2.GaussianBlur(img, (k, k), 0)

def down_up(img, scale):
    h, w = img.shape[:2]
    small = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    back  = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    return back

def contrast_scale(img, a):
    x = img.astype(np.float32) * a
    return np.clip(x, 0, 255).astype(np.uint8)

def gamma_corr(img, gamma):
    x = img.astype(np.float32) / 255.0
    y = np.power(x, gamma)
    return np.clip(y*255, 0, 255).astype(np.uint8)

settings = [
    ("clean", None),
    ("noise_sigma10", ("noise", 10)),
    ("noise_sigma25", ("noise", 25)),
    ("blur_k3", ("blur", 3)),
    ("blur_k7", ("blur", 7)),
    ("down_up_0.5", ("downup", 0.5)),
    ("down_up_0.25", ("downup", 0.25)),
    ("contrast_0.8", ("contrast", 0.8)),
    ("contrast_1.2", ("contrast", 1.2)),
    ("gamma_0.8", ("gamma", 0.8)),
    ("gamma_1.2", ("gamma", 1.2)),
]

img_files = sorted(os.listdir(img_dir))
rows = []

for f in tqdm(img_files):
    img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f)), cv2.COLOR_BGR2RGB)
    gt  = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)
    box = mask_to_box(gt)

    for tag, spec in settings:
        if spec is None:
            x = img
        else:
            kind, val = spec
            if kind == "noise":
                x = add_gaussian_noise(img, val)
            elif kind == "blur":
                x = gaussian_blur(img, val)
            elif kind == "downup":
                x = down_up(img, val)
            elif kind == "contrast":
                x = contrast_scale(img, val)
            elif kind == "gamma":
                x = gamma_corr(img, val)

        predictor.set_image(x)
        masks, scores, _ = predictor.predict(box=box, multimask_output=True)
        pred = masks[np.argmax(scores)]

        d, j = dice_iou(pred, gt)
        rows.append({"file": f, "condition": tag, "dice": d, "iou": j})

rob_df = pd.DataFrame(rows)
rob_df.to_csv("spleen_sam_robustness_results.csv", index=False)
print("✅ Saved spleen_sam_robustness_results.csv")

In [ ]:
import pandas as pd

rob_df = pd.read_csv("spleen_sam_robustness_results.csv")

base = rob_df[rob_df["condition"]=="clean"][["file","dice","iou"]].rename(columns={"dice":"dice_clean","iou":"iou_clean"})
merged = rob_df.merge(base, on="file")
merged["delta_dice"] = merged["dice"] - merged["dice_clean"]
merged["delta_iou"]  = merged["iou"] - merged["iou_clean"]

summary = merged.groupby("condition").agg(
    mean_dice=("dice","mean"),
    mean_iou=("iou","mean"),
    mean_delta_dice=("delta_dice","mean"),
    mean_delta_iou=("delta_iou","mean"),
    failure_rate_05=("dice", lambda s: (s<0.5).mean())
).reset_index().sort_values("mean_delta_dice")

print(summary)
summary.to_csv("robustness_summary_table.csv", index=False)
print("✅ Saved robustness_summary_table.csv")

In [ ]:
import pandas as pd

rob_df = pd.read_csv("spleen_sam_robustness_FAST.csv")

# Separate clean baseline
base = rob_df[rob_df["condition"]=="clean"][["file","dice","iou"]]
base = base.rename(columns={"dice":"dice_clean","iou":"iou_clean"})

# Merge back
merged = rob_df.merge(base, on="file")

# Compute deltas
merged["delta_dice"] = merged["dice"] - merged["dice_clean"]
merged["delta_iou"]  = merged["iou"] - merged["iou_clean"]

# Aggregate
summary = merged.groupby("condition").agg(
    Mean_Dice=("dice","mean"),
    Mean_IoU=("iou","mean"),
    Mean_Delta_Dice=("delta_dice","mean"),
    Mean_Delta_IoU=("delta_iou","mean"),
    Failure_Rate_05=("dice", lambda s: (s<0.5).mean())
).reset_index()

summary = summary.sort_values("Mean_Delta_Dice")
summary.to_csv("robustness_summary_table_FAST.csv", index=False)

print(summary)
print("\n✅ Saved robustness_summary_table_FAST.csv")

In [ ]:
!ls

In [ ]:
!ls *.csv

In [ ]:
# Separate clean baseline
base = rob_df[rob_df["condition"]=="clean"][["file","dice","iou"]]
base = base.rename(columns={"dice":"dice_clean","iou":"iou_clean"})

# Merge back
merged = rob_df.merge(base, on="file")

# Compute deltas
merged["delta_dice"] = merged["dice"] - merged["dice_clean"]
merged["delta_iou"]  = merged["iou"] - merged["iou_clean"]

# Aggregate summary
summary = merged.groupby("condition").agg(
    Mean_Dice=("dice","mean"),
    Mean_IoU=("iou","mean"),
    Mean_Delta_Dice=("delta_dice","mean"),
    Mean_Delta_IoU=("delta_iou","mean"),
    Failure_Rate_05=("dice", lambda s: (s<0.5).mean())
).reset_index()

summary = summary.sort_values("Mean_Delta_Dice")

summary.to_csv("robustness_summary_table_FINAL.csv", index=False)

print(summary)
print("\n✅ Saved robustness_summary_table_FINAL.csv")

In [ ]:
import seaborn as sns
import pandas as pd

sns.barplot(data=robustness_df,
            x="perturbation",
            y="delta_dice",
            ci="sd")

plt.axhline(0, linestyle="--")
plt.ylabel("ΔDice")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figure3_robustness.png", dpi=300)

In [ ]:
good_slices = df[(df["dice"] >= 0.92) & (df["dice"] <= 0.95)]
print(len(good_slices))

In [ ]:
%who

In [ ]:
import os, sys, torch
print("Python:", sys.version)
print("cwd:", os.getcwd())
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Files here:", os.listdir(".")[:20])

In [ ]:
%who

In [ ]:
import os

project_path = "/content/drive/MyDrive/sam_project"
os.makedirs(project_path, exist_ok=True)

print("Project folder ready.")

In [ ]:
print(type(out_msk))
print(type(out_msk[0]))
print(type(out_img[0]))

In [ ]:
print(type(out_msk))
print(type(out_msk[0]))
print(type(out_img[0]))

<class 'str'>
<class 'str'>
<class 'str'>


In [ ]:
import cv2
import numpy as np

pred_masks = []
gt_masks = []

for pred_path, gt_path in zip(out_msk, out_img):
    pred = cv2.imread(pred_path, 0)   # grayscale
    gt   = cv2.imread(gt_path, 0)

    pred = (pred > 0).astype(np.uint8)
    gt   = (gt > 0).astype(np.uint8)

    pred_masks.append(pred)
    gt_masks.append(gt)

print("Loaded:", len(pred_masks))

TypeError: '>' not supported between instances of 'NoneType' and 'int'

In [ ]:
import pandas as pd

df = pd.read_csv("spleen_sam_boxprompt_metrics.csv")
print(df.head())
print(df["dice"].describe())

FileNotFoundError: [Errno 2] No such file or directory: 'spleen_sam_boxprompt_metrics.csv'

In [ ]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir(".")[:50])

Current folder: /content
Files here: ['.config', 'drive', 'Task09_Spleen.tar', 'Task09_Spleen', 'data', '._Task09_Spleen', 'sample_data']


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "spleen" in file and file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
rows.append({"file": f, "dice": d, "iou": j})
df = pd.DataFrame(rows)
df.to_csv("spleen_sam_boxprompt_metrics.csv", index=False)

NameError: name 'rows' is not defined

In [ ]:
import os

count = 0
for root, dirs, files in os.walk("/content/Task09_Spleen"):
    for f in files:
        if f.startswith("._"):
            os.remove(os.path.join(root, f))
            count += 1

print("Deleted mac metadata files:", count)

Deleted mac metadata files: 106


In [ ]:
img_files = sorted([
    f for f in os.listdir(image_dir)
    if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")
])

print("Total valid image files:", len(img_files))
print("First 3 valid images:", img_files[:3])

for img_fname in tqdm(img_files):
    ...

Total valid image files: 41
First 3 valid images: ['spleen_10.nii.gz', 'spleen_12.nii.gz', 'spleen_13.nii.gz']


100%|██████████| 41/41 [00:00<00:00, 429916.16it/s]


In [ ]:
import os
image_dir = "/content/Task09_Spleen/imagesTr"
label_dir = "/content/Task09_Spleen/labelsTr"

img_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")])
lbl_files = set([f for f in os.listdir(label_dir) if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")])

missing = [f for f in img_files if f not in lbl_files]
print("Missing labels for images:", len(missing))
print("Example missing:", missing[:5])

Missing labels for images: 0
Example missing: []


In [ ]:
import os

out_img = "/content/data/images"
out_msk = "/content/data/masks"

print("out_img exists:", os.path.exists(out_img))
print("out_msk exists:", os.path.exists(out_msk))

print("PNG images count:", len(os.listdir(out_img)) if os.path.exists(out_img) else 0)
print("PNG masks  count:", len(os.listdir(out_msk)) if os.path.exists(out_msk) else 0)

if os.path.exists(out_img):
    print("First 5 images:", sorted(os.listdir(out_img))[:5])
if os.path.exists(out_msk):
    print("First 5 masks :", sorted(os.listdir(out_msk))[:5])

out_img exists: True
out_msk exists: True
PNG images count: 0
PNG masks  count: 0
First 5 images: []
First 5 masks : []


In [ ]:
import nibabel as nib
import numpy as np
import os

image_dir = "/content/Task09_Spleen/imagesTr"
label_dir = "/content/Task09_Spleen/labelsTr"

test = sorted([f for f in os.listdir(image_dir) if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")])[0]

img = nib.load(os.path.join(image_dir, test)).get_fdata()
msk = nib.load(os.path.join(label_dir, test)).get_fdata()

print("img shape:", img.shape, "msk shape:", msk.shape)
print("mask total sum:", np.sum(msk))
print("non-empty slices:", np.sum([msk[:,:,z].sum() > 0 for z in range(msk.shape[2])]))

img shape: (512, 512, 55) msk shape: (512, 512, 55)
mask total sum: 53071.0
non-empty slices: 17


In [ ]:
import os, cv2
import nibabel as nib
import numpy as np

image_dir = "/content/Task09_Spleen/imagesTr"
label_dir = "/content/Task09_Spleen/labelsTr"

out_img = "/content/data/images"
out_msk = "/content/data/masks"
os.makedirs(out_img, exist_ok=True)
os.makedirs(out_msk, exist_ok=True)

def window_ct(x, level=50, width=400):
    lo = level - width/2
    hi = level + width/2
    x = np.clip(x, lo, hi)
    x = (x - lo) / (hi - lo + 1e-8)
    return (x * 255).astype(np.uint8)

img_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")])
test = img_files[0]

img = nib.load(os.path.join(image_dir, test)).get_fdata()
msk = nib.load(os.path.join(label_dir, test)).get_fdata()

saved = 0
for z in range(img.shape[2]):
    m = msk[:, :, z]
    if m.sum() == 0:
        continue

    x = window_ct(img[:, :, z])
    x_rgb = np.stack([x, x, x], axis=-1)

    stem = test.replace(".nii.gz", "")
    out_name = f"{stem}_z{z:03d}.png"

    ok1 = cv2.imwrite(os.path.join(out_img, out_name), x_rgb)
    ok2 = cv2.imwrite(os.path.join(out_msk, out_name), (m > 0).astype(np.uint8) * 255)

    print("Saved", out_name, "img:", ok1, "msk:", ok2)
    saved += 1
    if saved >= 5:
        break

print("✅ Saved test PNGs:", saved)
print("Now images count:", len(os.listdir(out_img)))
print("Now masks  count:", len(os.listdir(out_msk)))

Saved spleen_10_z023.png img: True msk: True
Saved spleen_10_z024.png img: True msk: True
Saved spleen_10_z025.png img: True msk: True
Saved spleen_10_z026.png img: True msk: True
Saved spleen_10_z027.png img: True msk: True
✅ Saved test PNGs: 5
Now images count: 5
Now masks  count: 5


In [ ]:
os.makedirs(out_img, exist_ok=True)
os.makedirs(out_msk, exist_ok=True)

In [ ]:
import os, cv2
import nibabel as nib
import numpy as np
from tqdm import tqdm

image_dir = "/content/Task09_Spleen/imagesTr"
label_dir = "/content/Task09_Spleen/labelsTr"

out_img = "/content/data/images"
out_msk = "/content/data/masks"
os.makedirs(out_img, exist_ok=True)
os.makedirs(out_msk, exist_ok=True)

def window_ct(x, level=50, width=400):
    lo = level - width/2
    hi = level + width/2
    x = np.clip(x, lo, hi)
    x = (x - lo) / (hi - lo + 1e-8)
    return (x * 255).astype(np.uint8)

img_files = sorted([
    f for f in os.listdir(image_dir)
    if f.endswith(".nii.gz") and not f.startswith("._") and not f.startswith(".")
])

saved = 0
cases_done = 0

for img_fname in tqdm(img_files, desc="Volumes"):
    img_path = os.path.join(image_dir, img_fname)
    lbl_path = os.path.join(label_dir, img_fname)

    # (You already confirmed missing labels=0, but keep safe)
    if not os.path.exists(lbl_path):
        continue

    img = nib.load(img_path).get_fdata()
    msk = nib.load(lbl_path).get_fdata()

    for z in range(img.shape[2]):
        m = msk[:, :, z]
        if m.sum() == 0:
            continue

        x = window_ct(img[:, :, z])
        x_rgb = np.stack([x, x, x], axis=-1)

        stem = img_fname.replace(".nii.gz", "")
        out_name = f"{stem}_z{z:03d}.png"

        ok1 = cv2.imwrite(os.path.join(out_img, out_name), x_rgb)
        ok2 = cv2.imwrite(os.path.join(out_msk, out_name), (m > 0).astype(np.uint8) * 255)

        if ok1 and ok2:
            saved += 1

    cases_done += 1

print("✅ Volumes processed:", cases_done)
print("✅ Non-empty PNG slices saved:", saved)
print("Images folder count:", len(os.listdir(out_img)))
print("Masks  folder count:", len(os.listdir(out_msk)))

Volumes: 100%|██████████| 41/41 [00:43<00:00,  1.06s/it]

✅ Volumes processed: 41
✅ Non-empty PNG slices saved: 1051
Images folder count: 1051
Masks  folder count: 1051


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
dst_root = "/content/drive/MyDrive/sam_project"
os.makedirs(dst_root, exist_ok=True)

# copy folders (can take a few minutes)
shutil.copytree("/content/data/images", f"{dst_root}/images", dirs_exist_ok=True)
shutil.copytree("/content/data/masks",  f"{dst_root}/masks",  dirs_exist_ok=True)

print("✅ Copied images/ and masks/ to Drive")

ValueError: Mountpoint must not already contain files

In [ ]:
import shutil

shutil.make_archive("sam_outputs", "zip", "/content/data")
print("ZIP created")

ZIP created


In [ ]:
from google.colab import files
files.download("sam_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

image_dir = "/content/data/images"
mask_dir  = "/content/data/masks"

files = sorted(os.listdir(image_dir))

rows = []

for f in files:
    img = cv2.imread(os.path.join(image_dir, f))
    pred = cv2.imread(os.path.join(mask_dir, f), 0)
    gt   = cv2.imread(os.path.join(mask_dir, f), 0)  # if same for now

    pred = (pred > 0).astype(np.uint8)
    gt   = (gt > 0).astype(np.uint8)

    intersection = np.logical_and(pred, gt).sum()
    dice = (2 * intersection) / (pred.sum() + gt.sum() + 1e-8)

    rows.append({"file": f, "dice": dice})

df = pd.DataFrame(rows)
print(df.head())
print("Mean Dice:", df["dice"].mean())

                 file  dice
0  spleen_10_z023.png   1.0
1  spleen_10_z024.png   1.0
2  spleen_10_z025.png   1.0
3  spleen_10_z026.png   1.0
4  spleen_10_z027.png   1.0
Mean Dice: 0.9999999999918611


In [ ]:
df = pd.DataFrame(rows)

In [ ]:
rows = []
...
df = pd.DataFrame(rows)

In [ ]:
pred_dir = "/content/data/preds"
os.makedirs(pred_dir, exist_ok=True)

In [ ]:
# pred_mask is 0/1 numpy array
cv2.imwrite(os.path.join(pred_dir, out_name), (pred_mask.astype(np.uint8) * 255))

NameError: name 'pred_mask' is not defined

In [ ]:
pred_dir = "/content/data/preds"
os.makedirs(pred_dir, exist_ok=True)

rows = []

for f in sorted(os.listdir(image_dir)):
    ...
    predictor.set_image(image_rgb)

    masks, scores, logits = predictor.predict(
        box=box,
        multimask_output=False
    )

    pred_mask = masks[0]  # THIS is SAM output (0/1 numpy array)

    # Compute Dice immediately
    gt_mask = (m > 0).astype(np.uint8)
    pred_mask_bin = pred_mask.astype(np.uint8)

    intersection = np.logical_and(pred_mask_bin, gt_mask).sum()
    dice = (2 * intersection) / (pred_mask_bin.sum() + gt_mask.sum() + 1e-8)

    rows.append({"file": out_name, "dice": float(dice)})

    # Save predicted mask PNG
    cv2.imwrite(
        os.path.join(pred_dir, out_name),
        pred_mask_bin * 255
    )

NameError: name 'predictor' is not defined